# Visión por Computadora
## Clase 11 — Clasificación de imágenes a escala: muchas clases, datos desbalanceados y lectura de errores

**Unidad 3 · Detección, segmentación y clasificación de imágenes**
**Docente: Jorge Arevalo**

> **Laboratorio para casa.** En clase vimos la teoría y jugamos con el dashboard. Este cuaderno repite todo línea por línea para que lo corras tú, con la GPU T4 de Colab, y respondas el TU TURNO y la pregunta escrita.

Hasta la Clase 9 trabajamos con datos cómodos, CIFAR-10 con 5 000 fotos de cada clase o abejas y hormigas en cantidades parecidas. Los problemas reales casi nunca vienen así. En una clínica llegan cien resfríos por cada neumonía, en una fábrica salen miles de piezas buenas por cada pieza con fisura, y en una cámara de tránsito pasan muchísimos autos por cada camión. Hoy entrenamos un clasificador de diez clases con datos **desbalanceados** a propósito, vemos cómo saca un accuracy que parece excelente mientras ignora las clases raras, aprendemos a leer sus errores clase por clase y probamos cuatro maneras de corregirlo.

La idea se entiende con una clínica. Imagina que de cada 100 pacientes, 90 llegan con resfrío, 8 con gripe y 2 con neumonía. Un médico que dice "es resfrío" a todos, sin mirar a nadie, acierta 90 de 100, un **90 % de accuracy**, y sin embargo nunca encuentra una neumonía, que es justo el caso que importa. Ese médico no es tonto, es perezoso de una manera muy racional, apuesta siempre por lo más frecuente. Nuestras redes hacen exactamente lo mismo cuando les damos datos desbalanceados, y hoy vamos a pillarlas en el acto.

**Antes de empezar**, activa la GPU de Colab en el menú **Entorno de ejecución → Cambiar tipo de entorno de ejecución → T4 GPU → Guardar**. Después ejecuta las celdas en orden, de arriba hacia abajo. Todo el notebook tarda unos 5 minutos con GPU, y casi todo ese tiempo se va en descargar CIFAR-10 y en pasar sus fotos una vez por ResNet-18 en la Parte 4.

**Una regla para todo el laboratorio**, antes de ejecutar cada celda escribe en una línea qué crees que va a pasar, y después compáralo con lo que salió. Las cifras de referencia que aparecen en el texto son las de la corrida del docente en una T4, con la misma semilla y la misma receta, y en tu Colab pueden variar un poco en los decimales, porque la GPU no siempre suma en el mismo orden.

---

## Parte 0 — Preparación

Primero comprobamos que Colab nos dio una GPU.

In [ ]:
# torch es PyTorch, la librería con la que construimos y entrenamos las redes
import torch

# Preguntamos a PyTorch si encuentra una GPU con CUDA (la de Colab es una NVIDIA T4)
hay_gpu = torch.cuda.is_available()

# Si hay GPU, mostramos su nombre para confirmar que es la T4
if hay_gpu:
    # get_device_name(0) devuelve el nombre de la primera GPU disponible
    print("GPU encontrada:", torch.cuda.get_device_name(0))
# Si no hay GPU, avisamos cómo activarla, pero dejamos seguir
else:
    # Mensaje con la ruta exacta del menú de Colab para activar la GPU
    print("AVISO: no hay GPU. Ve a Entorno de ejecución → Cambiar tipo de entorno → T4 GPU,")
    # Recordamos que sin GPU solo conviene el modo rápido
    print("guarda y vuelve a ejecutar todo. Sin GPU usa RAPIDO = True o tardará mucho.")

Ahora traemos todas las herramientas y fijamos en un solo lugar, el diccionario `P`, las decisiones del experimento (semilla, cuántas fotos tiene cada clase, épocas, tasa de aprendizaje). Son exactamente los mismos valores que usó el docente, por eso tus cifras deberían parecerse a las suyas. Si algún día quieres comprobar en un par de minutos que todo corre, aunque no tengas GPU, pon `"RAPIDO": True`, pero esas cifras no sirven para comparar.

In [ ]:
# os sirve para manejar carpetas y rutas de archivos
import os
# time mide cuántos segundos tarda cada paso
import time
# random es el generador de azar de Python (lo fijamos para poder repetir resultados)
import random

# numpy es la librería de matrices de Python
import numpy as np
# matplotlib dibuja gráficos dentro del notebook
import matplotlib.pyplot as plt

# nn tiene las piezas de las redes: capas, funciones de pérdida, etc.
import torch.nn as nn
# F trae las versiones "función" de esas piezas (interpolate, softmax), sin pesos propios
import torch.nn.functional as F
# torchvision trae datasets y modelos ya entrenados
import torchvision
# models contiene ResNet-18 con sus pesos de ImageNet
from torchvision import models
# Matriz de confusión y métricas por clase (precisión, recall y F1), ya programadas y probadas
from sklearn.metrics import confusion_matrix, precision_recall_fscore_support

# Elegimos dónde calcular: en la GPU si existe, si no en el procesador
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# Media precisión (float16) acelera mucho la T4; en el procesador no se usa
USAR_AMP = device.type == "cuda"
# Mostramos la elección para que quede registrada en la salida
print("Calculando en:", device, "| media precisión:", USAR_AMP)

# P reúne TODAS las decisiones del experimento en un solo lugar (los mismos valores del docente)
P = {
    # True = prueba rápida en pocos minutos (cifras NO comparables); False = corrida normal
    "RAPIDO": False,
    # Semilla del azar: con el mismo número, los resultados se repiten
    "SEMILLA": 0,
    # Fotos de cada clase que separamos para validación (balanceada: todas las clases con las mismas)
    "N_VAL_POR_CLASE": 500,
    # Fotos que conserva la clase más frecuente (avión) en el entrenamiento desbalanceado
    "N_MAX": 4500,
    # Factor de desbalance: la clase más frecuente tiene IMB veces más fotos que la más rara
    "IMB": 100,
    # Fotos de prueba por clase en la prueba balanceada (CIFAR-10 trae 1 000 de cada una)
    "N_TEST_POR_CLASE": 1000,
    # Fotos de avión en la prueba realista; las demás clases bajan con la misma cola larga
    "N_TEST_MAX_REAL": 1000,
    # Lado al que agrandamos cada foto de 32 x 32 antes de dársela a ResNet-18 (el tamaño de ImageNet)
    "LADO_RED": 224,
    # Fotos que pasan juntas por la red al calcular características
    "LOTE_EXTRAER": 500,
    # Épocas de entrenamiento de cada cabeza
    "EP_CABEZA": 40,
    # Fotos por lote al entrenar la cabeza
    "BATCH_CABEZA": 256,
    # Tasa de aprendizaje de Adam (la misma de la Clase 8)
    "LR_CABEZA": 1e-3,
    # Weight decay (el freno L2 de la Clase 7)
    "WD_CABEZA": 5e-4,
    # Valores de τ (tau) que probaremos en la Parte 11
    "TAUS": [0.0, 0.25, 0.5, 0.75, 1.0, 1.25, 1.5, 1.75, 2.0],
}

# Si es la prueba rápida, achicamos todo para que termine en pocos minutos
if P["RAPIDO"]:
    # Menos fotos, fotos más chicas y menos épocas
    P.update({"N_VAL_POR_CLASE": 50, "N_MAX": 450, "N_TEST_POR_CLASE": 100, "N_TEST_MAX_REAL": 100,
              "LADO_RED": 64, "EP_CABEZA": 3})
    # Aviso bien visible
    print("MODO RÁPIDO: las cifras de esta corrida NO sirven para comparar.")

# Imprimimos la configuración completa para tenerla a la vista
for clave, valor in P.items():
    # Cada línea muestra una decisión y su valor
    print(f"{clave:>16} = {valor}")

In [ ]:
# Esta función fija TODAS las fuentes de azar a la vez
def fijar_semilla(s):
    # Azar de Python
    random.seed(s)
    # Azar de numpy
    np.random.seed(s)
    # Azar de PyTorch en el procesador
    torch.manual_seed(s)
    # Azar de PyTorch en la GPU (si no hay GPU, esta línea no hace nada)
    torch.cuda.manual_seed_all(s)

# Pedimos a la GPU que use siempre los mismos algoritmos, para que dos corridas den lo mismo
torch.backends.cudnn.deterministic = True
# Apagamos la búsqueda automática del algoritmo más rápido, que también mete variación
torch.backends.cudnn.benchmark = False

# Fijamos la semilla una primera vez
fijar_semilla(P["SEMILLA"])
# Confirmamos
print("Semilla fijada en", P["SEMILLA"])

# Colores del curso para todos los gráficos de hoy
TERRA, ACENTO, TINTA, GRIS, VERDE = "#D08A5E", "#B85042", "#2E2A26", "#A39A93", "#2F7A52"

---

## Parte 1 — Primero lo pequeño: el médico que dice "resfrío" a todos

Antes de tocar una sola foto, armamos la clínica de la introducción con números. Son 100 pacientes, 90 con resfrío, 8 con gripe y 2 con neumonía, y un médico perezoso que responde "resfrío" a cualquiera que entre. Vamos a medir su accuracy y a contar cuántas neumonías encontró.

In [ ]:
# Nombres de los tres diagnósticos posibles
DIAG = ["resfrío", "gripe", "neumonía"]
# Los 100 pacientes de la clínica: 90 con resfrío (0), 8 con gripe (1) y 2 con neumonía (2)
real = np.array([0] * 90 + [1] * 8 + [2] * 2)
# El médico perezoso dice "resfrío" (0) a los 100, sin mirar a nadie
perezoso = np.zeros(100, dtype=int)

# Accuracy: fracción de pacientes en los que acertó
acc_perezoso = (real == perezoso).mean()
# Neumonías que de verdad había
hay_neumonias = (real == 2).sum()
# Neumonías que encontró: pacientes que ERAN neumonía y él DIJO neumonía
encontro = ((real == 2) & (perezoso == 2)).sum()

# Mostramos los dos números
print(f"Accuracy del médico perezoso: {acc_perezoso:.0%}")
# El número que el accuracy no muestra
print(f"Neumonías encontradas: {encontro} de {hay_neumonias}")

**Cómo leerlo.** Deberías ver un **90 %** de accuracy y **0 de 2** neumonías encontradas. El 90 % no miente, de verdad acertó en 90 pacientes, pero esconde lo único que nos importaba. Fíjate de dónde sale ese 90 %, es simplemente la proporción de resfríos en la clínica, así que el médico no ganó nada con su "diagnóstico", lo habría sacado igual sin abrir los ojos. De aquí sale la primera regla del día, **en un problema desbalanceado, un accuracy solo significa algo si lo comparas con el de apostar siempre por la clase más frecuente**. Un 90 % donde la clase mayoritaria ya es el 90 % vale exactamente cero.

---

## Parte 2 — Primero lo pequeño: recall, precisión y F1 a mano

Ahora llega un segundo médico, uno atento, que sí examina a cada paciente. Se equivoca a veces, pero encuentra las dos neumonías. Vamos a ponerlo frente al perezoso con tres métricas que miran **clase por clase**, y las calculamos primero a mano, con la matriz de confusión, para que no sean una caja negra.

- **Recall** de una clase es la pregunta "de los que **eran** de esta clase, ¿a cuántos encontré?". Si había 2 neumonías y encontré 1, el recall de neumonía es 1/2 = 0,5.
- **Precisión** de una clase es la pregunta "de los que **dije** que eran de esta clase, ¿cuántos lo eran de verdad?". Si dije "neumonía" 3 veces y acerté 2, la precisión es 2/3 ≈ 0,67.
- **F1** junta las dos en un solo número (su media armónica) y solo sale alto si **las dos** son altas.
- **Macro-F1** es el promedio del F1 de todas las clases, donde cada clase pesa lo mismo, tenga 90 pacientes o 2.

In [ ]:
# El médico atento: armamos sus diagnósticos paciente por paciente
atento = np.concatenate([
    # De los 90 resfríos: dice resfrío a 84, gripe a 5 y neumonía a 1 (una falsa alarma)
    [0] * 84 + [1] * 5 + [2] * 1,
    # De las 8 gripes: dice resfrío a 2 y gripe a 6
    [0] * 2 + [1] * 6,
    # De las 2 neumonías: dice neumonía a las 2
    [2] * 2,
])


# Construye la matriz de confusión a mano: fila = lo que ERA, columna = lo que DIJO
def matriz_confusion(real, pred, k=3):
    # Empezamos con una matriz k x k llena de ceros
    M = np.zeros((k, k), dtype=int)
    # Recorremos los pacientes
    for r, p in zip(real, pred):
        # Sumamos uno en la casilla (lo que era, lo que dijo)
        M[r, p] += 1
    # Devolvemos la matriz
    return M


# Calcula recall, precisión y F1 de cada clase a partir de la matriz
def metricas_a_mano(M):
    # Aquí guardamos una fila de resultados por clase
    filas = []
    # Recorremos las clases
    for c in range(len(M)):
        # Aciertos de la clase c: la diagonal
        tp = M[c, c]
        # Recall: aciertos dividido entre todos los que ERAN c (la fila completa)
        recall = tp / M[c, :].sum()
        # Precisión: aciertos dividido entre todos los que DIJO c (la columna completa)
        precision = tp / M[:, c].sum() if M[:, c].sum() > 0 else 0.0
        # F1: media armónica; vale 0 si alguna de las dos es 0
        f1 = 2 * precision * recall / (precision + recall) if precision + recall > 0 else 0.0
        # Guardamos
        filas.append((recall, precision, f1))
    # Devolvemos como arreglo de numpy
    return np.array(filas)


# Recorremos los dos médicos
for nombre, pred in [("perezoso", perezoso), ("atento", atento)]:
    # Su matriz de confusión
    M = matriz_confusion(real, pred)
    # Sus métricas por clase
    m = metricas_a_mano(M)
    # Encabezado con accuracy y macro-F1
    print(f"\nMédico {nombre} · accuracy {(real == pred).mean():.0%} · macro-F1 {m[:, 2].mean():.2f}")
    # La matriz, fila por fila
    print("  matriz (fila = lo que era, columna = lo que dijo):")
    # Una fila por diagnóstico real
    for c in range(3):
        # Fila de la matriz
        print(f"    {DIAG[c]:>9}: {M[c]}")
    # Las métricas de cada clase
    for c in range(3):
        # Recall, precisión y F1
        print(f"  {DIAG[c]:>9} → recall {m[c, 0]:.2f} · precisión {m[c, 1]:.2f} · F1 {m[c, 2]:.2f}")

# Comprobación: sklearn debe dar exactamente lo mismo que nuestra cuenta a mano
_, _, f1_sk, _ = precision_recall_fscore_support(real, atento, labels=[0, 1, 2], zero_division=0)
# Si coinciden, nuestra fórmula está bien
print(f"\nmacro-F1 del atento según sklearn: {f1_sk.mean():.2f} (debe coincidir con la cuenta a mano)")

**Cómo leerlo.** El perezoso saca **90 %** de accuracy y el atento **92 %**, apenas dos puntos de diferencia, y si solo miraras ese número dirías que son casi igual de buenos. El macro-F1 cuenta otra historia, **0,32** para el perezoso contra **0,80** para el atento, porque el perezoso tiene F1 cero en gripe y en neumonía, y el macro-F1 le da a cada una de esas clases el mismo peso que al resfrío.

Mira también la fila de neumonía del atento. Su recall es **1,00**, encontró las dos, pero su precisión es **0,67**, porque una vez dijo "neumonía" a alguien que tenía resfrío. Esa es la tensión de toda la clase de hoy, **para encontrar más casos raros hay que atreverse a decirlos más seguido, y eso trae alguna falsa alarma**. En una clínica esa falsa alarma cuesta una radiografía extra, y la neumonía no detectada puede costar una vida, así que el intercambio vale la pena. Guarda esta idea, porque vamos a ver la misma tensión con fotos de camiones y aviones.

---

## Parte 3 — CIFAR-10 con cola larga y dos pruebas para el mismo modelo

Pasamos a fotos. Tomamos el CIFAR-10 de la Clase 9 y lo **desbalanceamos a propósito** con la receta que usan los papers de este tema (se llama *CIFAR-10-LT*, de *long tail*, cola larga). La primera clase, avión, conserva 4 500 fotos, y cada clase siguiente pierde la misma proporción hasta que la última, camión, se queda con 45, cien veces menos. Si dibujas cuántas fotos tiene cada clase, ordenadas de más a menos, el gráfico tiene una **cabeza** alta a la izquierda y una **cola** larga y delgada a la derecha, y de ahí sale el nombre.

Y aquí está el truco del día, cada modelo lo vamos a medir en **dos pruebas distintas**:

- La **prueba realista** tiene las fotos repartidas con la misma cola larga del entrenamiento (1 000 aviones y solo 10 camiones), que es como llegarían las fotos en el mundo real.
- La **prueba balanceada** tiene 1 000 fotos de cada clase, así que cada clase pesa lo mismo y nadie se puede esconder.

In [ ]:
# Carpeta donde guardaremos los datasets
RAIZ = "./data"
# Creamos la carpeta si todavía no existe
os.makedirs(RAIZ, exist_ok=True)

# Descargamos (si hace falta) y abrimos el entrenamiento de CIFAR-10 (tarda un minuto la primera vez)
cifar_train = torchvision.datasets.CIFAR10(RAIZ, train=True, download=True)
# Lo mismo para la prueba
cifar_test = torchvision.datasets.CIFAR10(RAIZ, train=False, download=True)

# Nombres de las diez clases en castellano, en el orden de CIFAR-10
CLASES = ["avión", "auto", "pájaro", "gato", "venado", "perro", "rana", "caballo", "barco", "camión"]
# Número de clases
K = len(CLASES)

# Todas las fotos de entrenamiento como un tensor de enteros [50000, 32, 32, 3]
X_train_u8 = torch.tensor(cifar_train.data)
# Sus etiquetas (números del 0 al 9)
y_train_all = torch.tensor(cifar_train.targets)
# Las fotos de prueba [10000, 32, 32, 3]
X_test_u8 = torch.tensor(cifar_test.data)
# Sus etiquetas
y_test_all = torch.tensor(cifar_test.targets)


# Cuántas fotos le tocan a cada clase en una cola larga que empieza en n_max y cae 'imb' veces
def cuantas_por_clase(n_max, imb):
    # La clase c conserva n_max · (1/imb)^(c/9): la 0 se queda con todo y la 9 con n_max/imb (al menos 1)
    return [max(1, int(n_max * (1 / imb) ** (c / (K - 1)))) for c in range(K)]


# Generador con semilla, solo para barajar las fotos de cada clase
g = torch.Generator().manual_seed(P["SEMILLA"])
# Índices de validación (balanceada) y "reserva" ordenada de cada clase
IDX_VAL, RESERVA = [], {}
# Recorremos las diez clases
for c in range(K):
    # Posiciones de las fotos de la clase c en el entrenamiento original
    pos = torch.where(y_train_all == c)[0]
    # Las barajamos con la semilla
    pos = pos[torch.randperm(len(pos), generator=g)]
    # Las primeras van a validación
    IDX_VAL += pos[:P["N_VAL_POR_CLASE"]].tolist()
    # Las siguientes forman la reserva de la clase, siempre en el mismo orden
    RESERVA[c] = pos[P["N_VAL_POR_CLASE"]:P["N_VAL_POR_CLASE"] + P["N_MAX"]].tolist()

# Lo mismo para la prueba: fotos de cada clase barajadas con la semilla
PRUEBA_POR_CLASE = {}
# Recorremos las clases
for c in range(K):
    # Posiciones de la clase c en la prueba
    pos = torch.where(y_test_all == c)[0]
    # Barajadas
    pos = pos[torch.randperm(len(pos), generator=g)]
    # Nos quedamos con las que pide la configuración (las 1 000)
    PRUEBA_POR_CLASE[c] = pos[:P["N_TEST_POR_CLASE"]].tolist()
# La prueba balanceada son todas esas fotos juntas, en orden
IDX_TEST_BAL = sorted(sum(PRUEBA_POR_CLASE.values(), []))


# Índices del entrenamiento con cola larga para un factor de desbalance
def indices_entrenamiento(imb):
    # Fotos que le tocan a cada clase
    n = cuantas_por_clase(P["N_MAX"], imb)
    # Las primeras n[c] de cada reserva
    return sum([RESERVA[c][:n[c]] for c in range(K)], []), n


# Índices de la prueba realista para un factor de desbalance (es un pedazo de la balanceada)
def indices_realista(imb):
    # Fotos de prueba que le tocan a cada clase con la misma cola larga
    m = cuantas_por_clase(P["N_TEST_MAX_REAL"], imb)
    # Las primeras m[c] de cada clase
    return sum([PRUEBA_POR_CLASE[c][:m[c]] for c in range(K)], []), m


# El entrenamiento principal de hoy (desbalance 100)
IDX_TR, N_TR = indices_entrenamiento(P["IMB"])
# Y su prueba realista
IDX_REAL, N_REAL = indices_realista(P["IMB"])

# Tabla del reparto
print(f"{'clase':>8} | {'entrenamiento':>13} | {'% del total':>11} | {'prueba realista':>15} | {'prueba balanceada':>17}")
# Separador
print("-" * 76)
# Una fila por clase
for c in range(K):
    # Fila
    print(f"{CLASES[c]:>8} | {N_TR[c]:>13} | {100 * N_TR[c] / sum(N_TR):>10.1f}% | {N_REAL[c]:>15} | {len(PRUEBA_POR_CLASE[c]):>17}")
# Totales
print(f"{'TOTAL':>8} | {sum(N_TR):>13} | {'100.0%':>11} | {sum(N_REAL):>15} | {len(IDX_TEST_BAL):>17}")

**Cómo leerlo.** Deberían salir **11 165** fotos de entrenamiento. Avión tiene 4 500, el **40,3 %** de todo, auto 2 697 y pájaro 1 617, y entre esas tres clases ya suman casi el 80 % del entrenamiento. En la otra punta, caballo tiene 125, barco 75 y camión apenas **45**, el 0,4 %. La prueba realista copia esa forma con 2 478 fotos, 1 000 aviones y **10 camiones**, y la balanceada tiene sus 1 000 de cada clase.

Fíjate en algo que va a importar mucho. Por casualidad, el orden de CIFAR-10 pone las clases raras justo al lado de clases frecuentes que se les parecen, **camión** (45 fotos) se parece a **auto** (2 697), **barco** (75) se parece a **avión** (4 500) y **caballo** (125) se parece a **venado** (581). En un problema real pasa lo mismo, el defecto raro de una pieza se parece a una pieza buena con una mancha, y la neumonía se parece a una gripe fuerte.

In [ ]:
# Dibujamos dos fotos de cada clase y, debajo, la barra de cuántas fotos tiene esa clase
fig = plt.figure(figsize=(15, 5.2))
# Rejilla: dos filas de fotos y una fila más alta para las barras
rej = fig.add_gridspec(3, K, height_ratios=[1, 1, 1.6], hspace=0.35)
# Recorremos las clases
for c in range(K):
    # Las dos primeras fotos de la reserva de la clase c
    for fila, i in enumerate(RESERVA[c][:2]):
        # Un recuadro para cada foto
        ax = fig.add_subplot(rej[fila, c])
        # Mostramos la foto de 32 x 32 sin suavizar los píxeles
        ax.imshow(X_train_u8[i].numpy(), interpolation="nearest")
        # El nombre de la clase solo encima de la primera foto
        ax.set_title(CLASES[c] if fila == 0 else "", fontsize=10)
        # Sin ejes
        ax.axis("off")
# Fila de abajo: barras con las fotos de entrenamiento de cada clase
ax = fig.add_subplot(rej[2, :])
# Las tres primeras clases en gris (la cabeza) y las tres últimas en terracota (la cola)
colores = [GRIS if c < 7 else ACENTO for c in range(K)]
# Barras
ax.bar(CLASES, N_TR, color=colores)
# Número encima de cada barra
for c in range(K):
    # Texto centrado sobre la barra
    ax.text(c, N_TR[c] + 60, f"{N_TR[c]:,}".replace(",", " "), ha="center", fontsize=10, color=TINTA)
# Título del eje
ax.set_ylabel("fotos de entrenamiento")
# Sin bordes de arriba y derecha
ax.spines[["top", "right"]].set_visible(False)
# Título general
fig.suptitle("La cola larga: la cabeza tiene miles de fotos, la cola apenas decenas", fontsize=13)
# Mostramos
plt.show()

**Cómo leerlo.** Las fotos se ven igual de claras en todas las clases, un camión es tan reconocible como un avión. Lo único que cambia es **cuántas** fotos de cada clase ve la red, y en la barra de abajo la diferencia es brutal, la barra de avión es cien veces más alta que la de camión. Todo lo que pase hoy viene de esa única diferencia.

---

## Parte 4 — El cuerpo congelado: 512 números por foto, una sola vez

Usamos la receta que ganó en la Clase 8, **ResNet-18 preentrenada en ImageNet con el cuerpo congelado** (*feature extraction*). Como el cuerpo no cambia, cada foto produce siempre los mismos 512 números, así que los calculamos **una sola vez** para todas las fotos y después entrenamos solo la cabeza, una capa lineal de 512 a 10, que tarda un par de segundos. Eso nos permite comparar cinco estrategias distintas sin esperar horas.

Dos detalles de la Clase 8 que vuelven hoy. Agrandamos cada foto de 32 × 32 a 224 × 224, el tamaño con el que aprendió ResNet-18, y la normalizamos con la media y la desviación de **ImageNet**, no las de CIFAR, porque el cuerpo aprendió a ver con fotos de ImageNet y hay que hablarle en su idioma.

In [ ]:
# Media de cada canal en ImageNet (las fotos con las que aprendió el cuerpo)
MEDIA_IMAGENET = torch.tensor([0.485, 0.456, 0.406], device=device).view(1, 3, 1, 1)
# Desviación de cada canal en ImageNet
DESV_IMAGENET = torch.tensor([0.229, 0.224, 0.225], device=device).view(1, 3, 1, 1)

# ResNet-18 con los pesos oficiales de ImageNet
cuerpo = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
# Le quitamos la cabeza de 1 000 clases: ahora devuelve los 512 números del cuerpo
cuerpo.fc = nn.Identity()
# A la GPU y en modo evaluación (nada del cuerpo va a cambiar)
cuerpo = cuerpo.to(device).eval()
# En la GPU usamos el orden de memoria que la T4 procesa más rápido
if device.type == "cuda":
    # channels_last acomoda los canales al final de cada píxel
    cuerpo = cuerpo.to(memory_format=torch.channels_last)


# Pasa fotos enteras [N, 32, 32, 3] por el cuerpo y devuelve sus características [N, 512]
def extraer(X_u8):
    # Aquí juntamos los resultados de cada lote
    salidas = []
    # Sin gradientes: el cuerpo está congelado y no aprende nada
    with torch.no_grad():
        # Recorremos las fotos de LOTE_EXTRAER en LOTE_EXTRAER
        for i in range(0, len(X_u8), P["LOTE_EXTRAER"]):
            # A la GPU, reordenadas a [lote, 3, 32, 32] y como decimales entre 0 y 1
            x = X_u8[i:i + P["LOTE_EXTRAER"]].to(device).permute(0, 3, 1, 2).float().div(255)
            # Agrandamos a 224 x 224 con interpolación bilineal
            x = F.interpolate(x, size=(P["LADO_RED"], P["LADO_RED"]), mode="bilinear", align_corners=False)
            # Normalizamos canal por canal con las cifras de ImageNet
            x = (x - MEDIA_IMAGENET) / DESV_IMAGENET
            # Mismo orden de memoria que el cuerpo
            if device.type == "cuda":
                # Reacomodamos el lote
                x = x.contiguous(memory_format=torch.channels_last)
            # Media precisión en la GPU para ir más rápido
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=USAR_AMP):
                # Los 512 números de cada foto
                f = cuerpo(x)
            # Los guardamos en precisión completa
            salidas.append(f.float())
    # Unimos todos los lotes
    return torch.cat(salidas)


# Reloj
t0 = time.time()
# Todos los índices de las reservas, clase por clase
IDX_RESERVA = sum([RESERVA[c] for c in range(K)], [])
# Características de toda la reserva (de aquí salen los entrenamientos de hoy y de TU TURNO)
F_RESERVA = extraer(X_train_u8[IDX_RESERVA])
# Dónde quedó cada foto original dentro de F_RESERVA
POS_RESERVA = {idx: i for i, idx in enumerate(IDX_RESERVA)}
# Características y etiquetas de validación
F_VAL = extraer(X_train_u8[IDX_VAL]); Y_VAL = y_train_all[IDX_VAL].to(device)
# Características y etiquetas de la prueba balanceada
F_TEST = extraer(X_test_u8[IDX_TEST_BAL]); Y_TEST = y_test_all[IDX_TEST_BAL].to(device)
# Dónde quedó cada foto de prueba dentro de F_TEST
POS_TEST = {idx: i for i, idx in enumerate(IDX_TEST_BAL)}
# Filas de F_TEST que forman la prueba realista
FILAS_REAL = torch.tensor([POS_TEST[i] for i in IDX_REAL], device=device)


# Características y etiquetas de un entrenamiento dado por sus índices originales
def datos_entrenamiento(idx):
    # Filas de F_RESERVA que corresponden a esas fotos
    filas = torch.tensor([POS_RESERVA[i] for i in idx], device=device)
    # Devolvemos características y etiquetas, ya en la GPU
    return F_RESERVA[filas], y_train_all[idx].to(device)


# Características del entrenamiento principal (desbalance 100)
F_TR, Y_TR = datos_entrenamiento(IDX_TR)
# Resumen
print(f"Reserva {tuple(F_RESERVA.shape)} · validación {tuple(F_VAL.shape)} · prueba {tuple(F_TEST.shape)}")
# Tiempo
print(f"Calculadas en {time.time() - t0:.1f} s")

**Cómo leerlo.** Cada foto quedó convertida en una fila de **512 números**, y tienes 45 000 filas de reserva, 5 000 de validación y 10 000 de prueba. En la T4 del docente esto tardó **23,5 segundos**. Desde aquí ya no tocamos ninguna foto, todo lo que sigue es entrenar una capa de 512 × 10 = 5 120 pesos más 10 sesgos sobre esas filas, y cada entrenamiento dura un par de segundos.

Piensa en los 512 números como una ficha que el cuerpo llena para cada foto, con casillas del tipo "¿tiene ruedas?", "¿tiene fondo azul?", "¿tiene patas?". La cabeza solo aprende a leer esas fichas, y esa ficha es igual de buena para un camión que para un avión, porque ImageNet le enseñó a ver ambos. Lo que va a fallar hoy **no es la ficha, es la lectura**.

---

## Parte 5 — Las herramientas: medir y entrenar

Tres funciones que vamos a usar toda la clase. `medir` saca de una vez la matriz de confusión y todas las métricas, `entrenar_cabeza` entrena la capa lineal (y acepta pesos por clase y sobremuestreo, que usaremos en la Parte 10), y `evaluar` mide un modelo en **las dos pruebas** y lo anota en la tabla `RES`.

In [ ]:
# Mide un conjunto de predicciones contra las etiquetas reales
def medir(y_real, y_pred):
    # Pasamos ambos a numpy en el procesador
    y_real = y_real.cpu().numpy(); y_pred = y_pred.cpu().numpy()
    # Matriz de confusión 10 x 10: fila = lo que ERA, columna = lo que DIJO el modelo
    cm = confusion_matrix(y_real, y_pred, labels=list(range(K)))
    # Precisión, recall y F1 de cada clase (zero_division=0 evita avisos si una clase nunca se predice)
    prec, rec, f1, _ = precision_recall_fscore_support(y_real, y_pred, labels=list(range(K)), zero_division=0)
    # Todo junto en un diccionario
    return {"acc": float((y_real == y_pred).mean()),   # fracción de aciertos sobre todas las fotos
            "acc_bal": float(rec.mean()),               # promedio del recall: cada clase pesa lo mismo
            "f1_macro": float(f1.mean()),               # promedio del F1 de las diez clases
            "recall": rec, "precision": prec, "f1": f1, "cm": cm}


# Puntajes (logits) de una cabeza lineal: diez números por foto, el más alto es la clase elegida
def puntajes_de(cabeza, Fx):
    # Modo evaluación
    cabeza.eval()
    # Sin gradientes
    with torch.no_grad():
        # Diez puntajes por foto
        return cabeza(Fx).float()


# Entrena una cabeza lineal de 512 a 10 sobre características ya calculadas
def entrenar_cabeza(Ftr, ytr, pesos=None, sobremuestreo=False):
    # Misma semilla en todos los experimentos: así solo cambia lo que queremos comparar
    fijar_semilla(P["SEMILLA"])
    # La cabeza: una capa lineal (512 x 10 pesos + 10 sesgos = 5 130 parámetros)
    cabeza = nn.Linear(Ftr.shape[1], K).to(device)
    # Adam con weight decay, como en la Clase 8
    opt = torch.optim.Adam(cabeza.parameters(), lr=P["LR_CABEZA"], weight_decay=P["WD_CABEZA"])
    # Entropía cruzada; 'pesos' (si viene) hace que equivocarse en unas clases cueste más que en otras
    crit = nn.CrossEntropyLoss(weight=pesos)
    # Número de fotos de entrenamiento
    N = len(ytr)
    # Generador de azar propio, para barajar igual en cada experimento
    gen = torch.Generator(device=device).manual_seed(P["SEMILLA"])
    # Si hay sobremuestreo, cada foto sale con probabilidad 1 / (fotos de su clase)
    if sobremuestreo:
        # Fotos de cada clase
        conteo = torch.bincount(ytr, minlength=K).float()
        # Probabilidad de cada foto: así cada clase aparece, en promedio, lo mismo en cada época
        prob = (1.0 / conteo)[ytr]
    # Historial de validación época por época
    hist = {"val_f1": [], "val_recall_cola": []}
    # Recorremos las épocas
    for ep in range(P["EP_CABEZA"]):
        # Modo entrenamiento
        cabeza.train()
        # Orden de las fotos en esta época
        if sobremuestreo:
            # N fotos sacadas con reemplazo según 'prob' (las raras salen repetidas muchas veces)
            orden = torch.multinomial(prob, N, replacement=True, generator=gen)
        else:
            # Una permutación normal de las N fotos
            orden = torch.randperm(N, device=device, generator=gen)
        # Recorremos los lotes
        for i in range(0, N, P["BATCH_CABEZA"]):
            # Índices del lote
            b = orden[i:i + P["BATCH_CABEZA"]]
            # Pérdida del lote
            loss = crit(cabeza(Ftr[b]), ytr[b])
            # Borramos los gradientes del paso anterior
            opt.zero_grad(set_to_none=True)
            # Calculamos los gradientes
            loss.backward()
            # Damos el paso
            opt.step()
        # Medimos en validación balanceada al final de la época
        mv = medir(Y_VAL, puntajes_de(cabeza, F_VAL).argmax(1))
        # Anotamos macro-F1 y recall de camión (la clase más rara)
        hist["val_f1"].append(mv["f1_macro"]); hist["val_recall_cola"].append(mv["recall"][K - 1])
    # Devolvemos la cabeza entrenada y su historial
    return cabeza, hist


# Tabla con los resultados de todos los experimentos
RES = {}


# Mide un modelo (dado por sus puntajes sobre la prueba) en las dos pruebas y lo anota
def evaluar(nombre, puntajes_prueba, mostrar=True):
    # Predicción: la clase con mayor puntaje
    pred = puntajes_prueba.argmax(1)
    # Métricas en la prueba balanceada
    mb = medir(Y_TEST, pred)
    # Métricas en la prueba realista (es un pedazo de la balanceada: no hace falta predecir de nuevo)
    mr = medir(Y_TEST[FILAS_REAL], pred[FILAS_REAL])
    # Anotamos
    RES[nombre] = {"acc_real": mr["acc"], "acc_bal": mb["acc_bal"], "f1_macro": mb["f1_macro"],
                   "recall": mb["recall"], "precision": mb["precision"], "f1": mb["f1"],
                   "cm": mb["cm"], "cm_real": mr["cm"], "pred": pred}
    # Una línea de resumen
    if mostrar:
        # Resumen
        r = RES[nombre]
        print(f"{nombre:>14} | prueba realista {r['acc_real']:.1%} | prueba balanceada {r['acc_bal']:.1%} | "
              f"macro-F1 {r['f1_macro']:.3f} | recall avión {r['recall'][0]:.0%} · camión {r['recall'][K - 1]:.0%}")

# Confirmamos
print("Herramientas listas.")

Una aclaración sobre la **prueba balanceada**. Ahí reportamos el *accuracy balanceado*, que es simplemente el promedio del recall de las diez clases. Como esa prueba tiene 1 000 fotos de cada clase, ese número coincide con su accuracy de siempre, la diferencia es solo de nombre. Lo importante es que ahí cada clase pesa un 10 %, mientras que en la prueba realista avión pesa un 40 % y camión un 0,4 %.

---

## Parte 6 — La trampa más barata: el modelo que siempre dice "avión"

El médico perezoso, versión fotos. No entrena nada, responde "avión" a cualquier foto. Antes de ejecutar, apuesta, ¿cuánto accuracy saca en la prueba realista?

In [ ]:
# Puntajes del "modelo de la mayoría": 10 para avión (la clase 0) y 0 para todas las demás, en cada foto
puntajes_mayoria = F.one_hot(torch.zeros(len(F_TEST), dtype=torch.long, device=device), K).float() * 10
# Lo medimos en las dos pruebas
evaluar("mayoria", puntajes_mayoria)

**Cómo leerlo.** En la corrida del docente saca **40,4 %** en la prueba realista sin haber aprendido absolutamente nada, porque 1 000 de las 2 478 fotos son aviones. En la balanceada saca **10 %**, lo mismo que tirar un dado de diez caras, y su macro-F1 es **0,018**. Ese 40,4 % es el **piso** de hoy, cualquier modelo que presentemos tiene que compararse contra este número y no contra cero. Si en tu proyecto una clase es el 70 % de los datos, un accuracy de 72 % no es un logro, es casi el médico perezoso.

---

## Parte 7 — El modelo base: el que parece un éxito

Ahora sí entrenamos la cabeza, con la entropía cruzada de siempre, sobre las 11 165 fotos desbalanceadas. Es exactamente lo que haría cualquiera que no sabe que sus datos están desbalanceados. Apuesta antes de ejecutar, ¿cuánto sacará en la prueba realista y cuánto en la balanceada?

In [ ]:
# Reloj
t0 = time.time()
# Entrenamos la cabeza base, sin ninguna corrección
cab_base, hist_base = entrenar_cabeza(F_TR, Y_TR)
# Medimos en las dos pruebas
evaluar("base", puntajes_de(cab_base, F_TEST))
# Tiempo de entrenamiento
print(f"Entrenada en {time.time() - t0:.1f} s")

**Cómo leerlo.** En la corrida del docente el modelo base sacó **91,0 %** en la prueba realista. Es un número que en una exposición de avances suena a éxito rotundo, nueve de cada diez fotos bien clasificadas, y le gana por más de 50 puntos al modelo de la mayoría. Pero en la prueba balanceada baja a **74,2 %**, y su macro-F1 es **0,739**. Son **17 puntos de diferencia** para el mismo modelo, con los mismos pesos, solo cambiando qué fotos le mostramos al final. Ese salto es el engaño del accuracy, y ahora vamos a ver de dónde sale, clase por clase.

In [ ]:
# Tabla por clase del modelo base, en la prueba balanceada
r = RES["base"]
# Encabezado
print(f"{'clase':>8} | {'fotos train':>11} | {'recall':>6} | {'precisión':>9} | {'F1':>5}")
# Separador
print("-" * 52)
# Una fila por clase, de la más frecuente a la más rara
for c in range(K):
    # Fila
    print(f"{CLASES[c]:>8} | {N_TR[c]:>11} | {r['recall'][c]:>6.2f} | {r['precision'][c]:>9.2f} | {r['f1'][c]:>5.2f}")

**Cómo leerlo.** Lee la columna de **recall** de arriba hacia abajo y verás que cae casi en escalera junto con las fotos de entrenamiento. Avión encuentra el **98 %** de sus fotos y auto el 96 %, pero barco baja a **53 %** y camión a **43,5 %** (435 de 1 000), menos de la mitad de los camiones.

Ahora lee la columna de **precisión**, que hace justo lo contrario. Avión tiene precisión **0,56**, o sea que de cada 100 veces que el modelo dice "avión", 44 veces se equivoca, y camión tiene **1,00**, cuando el modelo dice "camión" acierta prácticamente siempre (en la corrida del docente, 0,995). Es el médico tímido de la clínica, que solo diagnostica neumonía cuando está segurísimo, y por eso nunca se equivoca cuando lo dice, pero deja pasar a la mitad de los enfermos. Y "avión" se volvió el cajón de sastre donde el modelo echa todo lo que no tiene claro. **Recall bajo con precisión alta es la firma de una clase que el modelo tiene miedo de nombrar.**

---

## Parte 8 — La matriz de confusión: dónde se van los errores

El recall nos dice **cuánto** falla cada clase, y la matriz de confusión nos dice **con qué** la confunde. Cada fila es lo que la foto **era**, cada columna lo que el modelo **dijo**, la diagonal son los aciertos y todo lo que queda fuera son errores con dirección. La dibujamos en porcentaje por fila, para que una clase con 1 000 fotos y otra con 1 000 fotos se lean igual.

In [ ]:
# Dibuja una matriz de confusión con el porcentaje de cada fila y el conteo en cada casilla
def dibujar_matriz(cm, titulo, ax):
    # Porcentaje por fila: cada fila suma 100 %
    cm_pct = cm / cm.sum(axis=1, keepdims=True)
    # Mapa de color de crema a terracota oscuro
    ax.imshow(cm_pct, cmap="Oranges", vmin=0, vmax=1)
    # Nombres de las clases en los dos ejes
    ax.set_xticks(range(K)); ax.set_xticklabels(CLASES, rotation=45, ha="right")
    ax.set_yticks(range(K)); ax.set_yticklabels(CLASES)
    # Títulos de los ejes
    ax.set_xlabel("lo que DIJO el modelo"); ax.set_ylabel("lo que ERA la foto")
    # Título del gráfico
    ax.set_title(titulo)
    # Escribimos el conteo en cada casilla que no sea cero
    for i in range(K):
        # Columnas
        for j in range(K):
            # Solo casillas con algo
            if cm[i, j] > 0:
                # Texto blanco sobre fondo oscuro, oscuro sobre fondo claro
                ax.text(j, i, cm[i, j], ha="center", va="center", fontsize=8,
                        color="white" if cm_pct[i, j] > 0.5 else TINTA)


# Una figura con la matriz del modelo base
fig, ax = plt.subplots(figsize=(8.5, 7.5))
# La dibujamos
dibujar_matriz(RES["base"]["cm"], "Modelo base · prueba balanceada (1 000 fotos por clase)", ax)
# Mostramos
plt.tight_layout(); plt.show()

# Suma de cada columna: cuántas veces dijo cada clase
dijo = RES["base"]["cm"].sum(axis=0)
# Lo imprimimos
print("Veces que el modelo dijo cada clase (había 1 000 de cada una):")
# Una línea por clase
for c in range(K):
    # Fila
    print(f"  {CLASES[c]:>8}: {dijo[c]:>5}")

**Cómo leerlo.** Busca la fila de **camión**. En la corrida del docente, de los 1 000 camiones solo 435 quedaron en la diagonal, **394 se fueron a la columna de auto** y 155 a la de avión. La fila de **barco** manda **375** fotos a avión. La fila de **perro** manda 209 a gato. Los errores no están repartidos al azar, cada clase rara se derrama hacia su vecina frecuente.

Ahora mira las sumas por columna. Había 1 000 aviones, pero el modelo dijo "avión" **1 740** veces, y dijo "auto" 1 424 veces. En cambio dijo "camión" solo 437 veces. El modelo aprendió algo muy razonable para su entrenamiento, que si una foto con ruedas le genera dudas, apostar por "auto" acierta mucho más seguido que apostar por "camión", porque vio 2 697 autos y 45 camiones. No es un error de programación, es **estadística bien aprendida de unos datos que no representan lo que nos importa**.

In [ ]:
# Mostramos 12 camiones que el modelo base confundió, con lo que dijo
pred_base = RES["base"]["pred"].cpu()
# Fotos de prueba que eran camión y el base NO dijo camión
errores_camion = [i for i in PRUEBA_POR_CLASE[K - 1] if pred_base[POS_TEST[i]].item() != K - 1][:12]
# Figura con 12 recuadros
fig, axes = plt.subplots(2, 6, figsize=(13, 4.8))
# Recorremos los recuadros
for ax, i in zip(axes.ravel(), errores_camion):
    # La foto original de 32 x 32
    ax.imshow(X_test_u8[i].numpy(), interpolation="nearest")
    # Lo que dijo el modelo, en rojo
    ax.set_title(f"dijo: {CLASES[pred_base[POS_TEST[i]].item()]}", fontsize=10, color=ACENTO)
    # Sin ejes
    ax.axis("off")
# Título general
fig.suptitle("Eran camiones, y el modelo base dijo otra cosa", fontsize=13)
# Mostramos
plt.tight_layout(); plt.show()

**Cómo leerlo.** Mira las fotos con calma. La mayoría son camiones perfectamente reconocibles para una persona, y el modelo dijo "auto" o "avión". Algunas tienen excusa (un camión de frente que parece un auto, uno sobre fondo de cielo que parece un avión), pero muchas no la tienen. Si el problema fuera que el cuerpo no sabe ver camiones, estas fotos serían raras o borrosas. No lo son. El cuerpo ve bien, y es la cabeza la que, ante la duda, empuja la respuesta hacia la clase que vio más veces.

---

## Parte 9 — Por qué el accuracy se dejó engañar

Juntamos las piezas en un solo gráfico. Para cada clase dibujamos su recall (en la prueba balanceada) contra cuántas fotos tuvo en el entrenamiento, con el eje horizontal en escala logarítmica, porque las fotos van de 45 a 4 500.

In [ ]:
# Figura con dos paneles
fig, (a1, a2) = plt.subplots(1, 2, figsize=(14, 4.8))
# Panel 1: recall contra fotos de entrenamiento
a1.scatter(N_TR, RES["base"]["recall"], s=80, color=ACENTO, zorder=3)
# El nombre junto a cada punto
for c in range(K):
    # Texto un poco a la derecha del punto
    a1.annotate(CLASES[c], (N_TR[c], RES["base"]["recall"][c]), xytext=(6, -4), textcoords="offset points", fontsize=10)
# Escala logarítmica en el eje horizontal
a1.set_xscale("log")
# Títulos de los ejes
a1.set_xlabel("fotos de entrenamiento de la clase (escala log)"); a1.set_ylabel("recall en la prueba balanceada")
# Título del panel
a1.set_title("Menos fotos, menos recall")
# Límites verticales
a1.set_ylim(0, 1.05)
# Panel 2: los tres números del modelo base y el de la mayoría
nombres = ["prueba realista", "prueba balanceada", "macro-F1"]
# Valores del base
v_base = [RES["base"]["acc_real"], RES["base"]["acc_bal"], RES["base"]["f1_macro"]]
# Valores de la mayoría
v_may = [RES["mayoria"]["acc_real"], RES["mayoria"]["acc_bal"], RES["mayoria"]["f1_macro"]]
# Posiciones de las barras
x = np.arange(3)
# Barras del base
a2.bar(x - 0.2, v_base, 0.4, color=ACENTO, label="modelo base")
# Barras de la mayoría
a2.bar(x + 0.2, v_may, 0.4, color=GRIS, label="siempre 'avión'")
# Valor encima de cada barra
for i in range(3):
    # Texto del base
    a2.text(i - 0.2, v_base[i] + 0.02, f"{v_base[i]:.2f}", ha="center", fontsize=10)
    # Texto de la mayoría
    a2.text(i + 0.2, v_may[i] + 0.02, f"{v_may[i]:.2f}", ha="center", fontsize=10)
# Etiquetas
a2.set_xticks(x); a2.set_xticklabels(nombres); a2.set_ylim(0, 1.08); a2.legend()
# Título del panel
a2.set_title("El mismo modelo, tres números distintos")
# Sin bordes de arriba y derecha
for a in (a1, a2):
    # Bordes
    a.spines[["top", "right"]].set_visible(False)
# Mostramos
plt.tight_layout(); plt.show()

**Cómo leerlo.** En el panel de la izquierda los puntos dibujan una diagonal bastante clara, a menos fotos de entrenamiento, menos recall, desde avión arriba a la derecha hasta camión abajo a la izquierda. En el de la derecha está el engaño completo. La prueba realista le da al base **0,91**, porque ahí avión, auto y pájaro son casi el 80 % de las fotos y esas clases el modelo las hace muy bien. La prueba balanceada y el macro-F1, donde camión vale lo mismo que avión, lo bajan a **0,74**.

Entonces, ¿cuál número es el verdadero? Los dos son verdaderos, responden preguntas distintas. El accuracy realista responde "¿cuántas fotos acierto si las fotos llegan como siempre?", y el macro-F1 responde "¿qué tan bien hago **cada** clase?". **Si las clases raras te importan, y en casi todo proyecto real son las que más importan, el número que tienes que mirar y reportar es el macro-F1, junto al recall de la clase rara.**

---

## Parte 10 — Tres curas que cambian el entrenamiento

El modelo base aprendió a apostar por las clases frecuentes porque así gana más puntos en la pérdida. Hay dos maneras clásicas de cambiar ese incentivo.

- **Reponderar la pérdida.** Le decimos a la entropía cruzada que equivocarse en una foto de camión cuesta más que equivocarse en una de avión. Con el **inverso de la frecuencia** ("puro"), cada error en camión pesa 24,8 y cada error en avión 0,25, unas cien veces más, así que en total las diez clases pesan lo mismo. Con la **raíz** del inverso la corrección es más suave, camión pesa 4,98 y avión 0,50, solo diez veces más.
- **Sobremuestrear la cola.** En vez de tocar la pérdida, cambiamos qué fotos ve la cabeza. Armamos cada época sacando fotos de manera que todas las clases aparezcan lo mismo, así que los 45 camiones salen repetidos muchísimas veces y los aviones aparecen menos.

Apuesta antes de ejecutar, ¿cuál de las tres sube más el recall de camión?

In [ ]:
# Pesos por clase para la pérdida: (N / (K · n_c)) ^ potencia
def pesos_por_clase(n, potencia):
    # Fotos de cada clase como tensor
    n = torch.tensor(n, dtype=torch.float32)
    # Peso de cada clase; potencia 1 = inverso exacto ("puro"), 0.5 = raíz (más suave), 0 = todos iguales
    w = (n.sum() / (K * n)) ** potencia
    # A la GPU (la escala no importa: la pérdida divide por la suma de los pesos usados)
    return w.to(device)


# Pesos de las dos curas de reponderación
W_RAIZ, W_PURO = pesos_por_clase(N_TR, 0.5), pesos_por_clase(N_TR, 1.0)
# Los mostramos
print(f"{'clase':>8} | {'peso raíz':>9} | {'peso puro':>9}")
# Una fila por clase
for c in range(K):
    # Fila
    print(f"{CLASES[c]:>8} | {W_RAIZ[c].item():>9.2f} | {W_PURO[c].item():>9.2f}")
print()

# Cura 1: pesos con raíz
cab_raiz, hist_raiz = entrenar_cabeza(F_TR, Y_TR, pesos=W_RAIZ)
# Medida
evaluar("raiz", puntajes_de(cab_raiz, F_TEST))
# Cura 2: pesos con inverso exacto
cab_puro, hist_puro = entrenar_cabeza(F_TR, Y_TR, pesos=W_PURO)
# Medida
evaluar("puro", puntajes_de(cab_puro, F_TEST))
# Cura 3: sobremuestreo de la cola
cab_sobre, hist_sobre = entrenar_cabeza(F_TR, Y_TR, sobremuestreo=True)
# Medida
evaluar("sobremuestreo", puntajes_de(cab_sobre, F_TEST))

**Cómo leerlo.** En la corrida del docente las tres curas mejoran al base, y en orden. La raíz sube el macro-F1 de 0,739 a **0,791** y el recall de camión de 43,5 % a **60 %**. El inverso puro llega más lejos, macro-F1 **0,826** y camión **73 %**. El sobremuestreo queda casi empatado con el puro, **0,817** y 68,5 %, lo que tiene sentido, porque repetir una foto 25 veces y hacer que su error cuente 25 veces son dos caminos hacia la misma idea.

Pero mira la columna que **baja**. El recall de avión pasa de 98 % en el base a 96 % con raíz y a **92 %** con el puro, y el accuracy de la prueba realista cae de 91,0 % a 90,4 % y a **88,7 %**. Ningún método regala nada, todos mueven la frontera, le quitan a la cabeza para darle a la cola. Es la misma tensión del médico atento de la Parte 2, para encontrar más neumonías hay que atreverse a decir "neumonía" más seguido.

In [ ]:
# Curva de validación época por época: macro-F1 y recall de camión del base y de las tres curas
fig, (a1, a2) = plt.subplots(1, 2, figsize=(14, 4.4))
# Épocas en el eje horizontal
ep = range(1, P["EP_CABEZA"] + 1)
# Una curva por experimento, con sus colores
for nombre, h, col in [("base", hist_base, GRIS), ("raíz", hist_raiz, TERRA), ("puro", hist_puro, ACENTO), ("sobremuestreo", hist_sobre, VERDE)]:
    # Macro-F1 en validación
    a1.plot(ep, h["val_f1"], color=col, lw=2.4, label=nombre)
    # Recall de camión en validación
    a2.plot(ep, h["val_recall_cola"], color=col, lw=2.4, label=nombre)
# Títulos y ejes
a1.set_title("macro-F1 en validación, época a época"); a2.set_title("recall de camión en validación, época a época")
# Ejes
for a in (a1, a2):
    # Etiqueta del eje
    a.set_xlabel("época"); a.legend(); a.set_ylim(0, 1); a.spines[["top", "right"]].set_visible(False)
# Mostramos
plt.tight_layout(); plt.show()

**Cómo leerlo.** En el panel de la derecha la curva gris del base sube muy despacio, se queda en 0 % de camiones las tres primeras épocas y termina en **43 %** después de 40, como si a la cabeza le costara mucho convencerse de que existe una clase con tan pocas fotos. La raíz sube más rápido y se estabiliza cerca del 60 %. Los pesos puros y el sobremuestreo hacen algo llamativo, arrancan con muchísimos camiones desde la primera época (76 % y 84 % en la corrida del docente) y después bajan un poco hasta estabilizarse, porque al principio la cabeza exagera la corrección.

En el panel de la izquierda, el macro-F1 del base y de la raíz se aplana hacia la época 10. El de los pesos puros y el sobremuestreo toca su techo temprano, en las épocas 14 y 6, y cede unos dos puntos hasta la época 40, como si al entrenar mucho la cabeza volviera a acomodarse un poco hacia las clases frecuentes. Por eso en tu proyecto conviene elegir la época mirando validación, el *early stopping* de la Clase 7. Lo que no vemos es la firma del sobreajuste, el problema del base no era entrenar de más ni de menos, era **hacia dónde** lo empujaban los datos.

---

## Parte 11 — Una cura que no reentrena nada: el ajuste de logits

La última idea es la más elegante. El modelo base, cuando ve una foto, en el fondo está mezclando dos preguntas en cada puntaje, "¿cuánto se parece esta foto a un camión?" y "¿cuán frecuentes son los camiones?". Esa segunda parte la aprendió de los datos, y es justo la que no queremos. Así que la **restamos**. A cada puntaje le quitamos τ · log(frecuencia de la clase), con la frecuencia que la clase tuvo en el entrenamiento.

Para ver qué hace esa resta, mira los números. La frecuencia de avión es 0,403 y su logaritmo −0,91, la de camión es 0,004 y su logaritmo −5,51. Con τ = 1, al puntaje de avión le restamos −0,91 (es decir, le **sumamos** 0,91) y al de camión le **sumamos** 5,51. Camión recibe una ventaja de 4,6 puntos sobre avión, que es exactamente la desventaja que le había dado haber visto cien veces menos fotos. Con τ = 0 no cambia nada, y con τ más grande la ventaja crece.

¿Qué τ usamos? Lo elegimos mirando **validación**, nunca la prueba. Si eligiéramos τ mirando la prueba, estaríamos haciendo trampa, igual que estudiar con las respuestas del examen.

In [ ]:
# Frecuencia de cada clase en el entrenamiento (la "creencia previa" que aprendió el modelo)
PRIOR = torch.tensor(N_TR, dtype=torch.float32, device=device) / sum(N_TR)
# Su logaritmo, que es lo que restaremos a los puntajes
LOG_PRIOR = PRIOR.log()
# Proporción de cada clase en la prueba realista, para estimar su accuracy desde el recall
PROP_REAL = torch.tensor(N_REAL, dtype=torch.float32) / sum(N_REAL)

# Puntajes del modelo base en VALIDACIÓN (no en la prueba)
S_VAL = puntajes_de(cab_base, F_VAL)
# Aquí guardamos la curva
curva = {"tau": [], "acc_bal": [], "acc_real_est": [], "recall_avion": [], "recall_camion": [], "f1": []}
# Probamos cada τ
for tau in P["TAUS"]:
    # Predicción con los puntajes corregidos: puntaje − τ · log(frecuencia)
    m = medir(Y_VAL, (S_VAL - tau * LOG_PRIOR).argmax(1))
    # Accuracy que tendría en una prueba realista: recall de cada clase por su proporción, sumado
    acc_real_est = float((torch.tensor(m["recall"], dtype=torch.float32) * PROP_REAL).sum())
    # Anotamos
    curva["tau"].append(tau); curva["acc_bal"].append(m["acc_bal"]); curva["acc_real_est"].append(acc_real_est)
    curva["recall_avion"].append(m["recall"][0]); curva["recall_camion"].append(m["recall"][K - 1]); curva["f1"].append(m["f1_macro"])
    # Fila
    print(f"τ = {tau:4.2f} | validación balanceada {m['acc_bal']:.1%} | realista estimada {acc_real_est:.1%} | "
          f"recall avión {m['recall'][0]:.0%} · camión {m['recall'][K - 1]:.0%}")

# Elegimos el τ con mejor macro-F1 en validación
TAU = curva["tau"][int(np.argmax(curva["f1"]))]
# Lo mostramos
print(f"\nτ elegido en validación: {TAU}")

# Dibujamos la curva
fig, ax = plt.subplots(figsize=(10, 4.6))
# Cuatro curvas contra τ
ax.plot(curva["tau"], curva["acc_bal"], "-o", color=ACENTO, lw=2.6, label="accuracy balanceado")
ax.plot(curva["tau"], curva["acc_real_est"], "-o", color=GRIS, lw=2.6, label="accuracy realista (estimado)")
ax.plot(curva["tau"], curva["recall_camion"], "--", color=TERRA, lw=2, label="recall camión")
ax.plot(curva["tau"], curva["recall_avion"], "--", color=TINTA, lw=2, label="recall avión")
# Línea vertical en el τ elegido
ax.axvline(TAU, color=VERDE, lw=1.5, ls=":"); ax.text(TAU + 0.03, 0.05, f"τ elegido = {TAU}", color=VERDE)
# Ejes y leyenda
ax.set_xlabel("τ"); ax.set_ylim(0, 1.02); ax.legend(loc="lower left"); ax.spines[["top", "right"]].set_visible(False)
# Título
ax.set_title("Cuánto τ, en validación: lo que gana la cola lo pierde la cabeza")
# Mostramos
plt.tight_layout(); plt.show()

**Cómo leerlo.** Con τ pasa como con la sal en la comida, ni poco ni mucho. Con **τ = 0** estás en el modelo base, recall de camión 43 % y de avión 97 %. Al subir τ, la línea del recall de camión trepa y la de avión baja, se cruzan cerca de **τ = 1** (los dos alrededor del 88-89 %), y en el docente el accuracy balanceado tocó su máximo, **84,9 %**, con **τ = 1,25**, que fue el elegido. Si sigues subiendo, con τ = 2 te pasas de la raya, camión llega a 96 % pero avión se desploma a **56 %**, y el balanceado vuelve a bajar.

Mira ahora la línea gris del accuracy realista, que hace lo contrario, **baja todo el tiempo**, de 91,2 % con τ = 0 a 64,2 % con τ = 2. Sobre esto la teoría dice algo bonito. Si un modelo está bien calibrado, τ = 1 es justo la corrección que deja a todas las clases en igualdad de condiciones (Menon et al., 2021), y por eso el óptimo cae tan cerca de 1.

In [ ]:
# Cura 4: aplicamos el τ elegido al modelo base, sobre la prueba, sin reentrenar nada
evaluar("ajuste_logits", puntajes_de(cab_base, F_TEST) - TAU * LOG_PRIOR)

**Cómo leerlo.** Con τ = 1,25 el docente obtuvo el mejor macro-F1 del día, **0,841**, y un recall de camión de **91 %**, más del doble que el base, sin entrenar ni un solo peso nuevo, solo restando un número fijo a cada puntaje. Pero el accuracy de la prueba realista bajó a **82,3 %**, casi 9 puntos menos que el base. En la prueba realista eso significa encontrar **9 de los 10 camiones** en vez de 3, a cambio de reconocer **831 de los 1 000 aviones** en vez de 981. ¿Vale la pena? Depende de cuánto cueste cada error en tu problema, y esa es una pregunta de negocio o de medicina, no de código.

---

## Parte 12 — Todos juntos: la paradoja del accuracy

Ponemos los seis modelos en una tabla y en un gráfico, con el accuracy de la prueba realista al lado del macro-F1.

In [ ]:
# Orden en que mostramos los experimentos
ORDEN = ["mayoria", "base", "raiz", "puro", "sobremuestreo", "ajuste_logits"]
# Tabla
print(f"{'experimento':>14} | {'realista':>8} | {'balanceada':>10} | {'macro-F1':>8} | {'recall avión':>12} | {'recall camión':>13}")
# Separador
print("-" * 82)
# Una fila por experimento
for n in ORDEN:
    # Resultado
    r = RES[n]
    # Fila
    print(f"{n:>14} | {r['acc_real']:>8.1%} | {r['acc_bal']:>10.1%} | {r['f1_macro']:>8.3f} | {r['recall'][0]:>12.0%} | {r['recall'][K - 1]:>13.0%}")

# Gráfico: dos barras por experimento
fig, ax = plt.subplots(figsize=(12, 4.6))
# Posiciones
x = np.arange(len(ORDEN))
# Accuracy realista
ax.bar(x - 0.2, [RES[n]["acc_real"] for n in ORDEN], 0.4, color=GRIS, label="accuracy en la prueba realista")
# Macro-F1
ax.bar(x + 0.2, [RES[n]["f1_macro"] for n in ORDEN], 0.4, color=ACENTO, label="macro-F1")
# Etiquetas
ax.set_xticks(x); ax.set_xticklabels(ORDEN); ax.set_ylim(0, 1.05); ax.legend(loc="upper left")
# Bordes
ax.spines[["top", "right"]].set_visible(False)
# Título
ax.set_title("Cuando el macro-F1 sube, el accuracy realista baja")
# Mostramos
plt.tight_layout(); plt.show()

**Cómo leerlo.** De izquierda a derecha, quitando la mayoría, las barras grises **bajan** (91,0 %, 90,4 %, 88,7 %, 88,7 %, 82,3 %) mientras las terracota **suben** (0,739, 0,791, 0,826, 0,817, 0,841). Es la paradoja de la clase, **el modelo que encuentra más camiones, más barcos y más caballos tiene un accuracy realista más bajo**. Si tu único criterio fuera el accuracy, elegirías el modelo base y te quedarías con un detector que deja pasar seis de cada diez camiones.

Para tu proyecto la lección es concreta. Reporta siempre el accuracy **junto con** el macro-F1 y el recall de la clase que más te importa, compara contra el modelo de la mayoría y muestra la matriz de confusión. Y antes de elegir una cura, decide qué error te duele más, porque cada cura es una manera distinta de repartir los errores entre la cabeza y la cola.

---

## Parte 13 — TU TURNO

Tienes tres palancas. Mueve **una sola a la vez**, ejecuta su celda y compara con las cifras de las Partes 7 a 12. Antes de ejecutar, escribe en el comentario qué esperas que pase.

**Palanca 1 · ¿Cuánto desbalance hace falta para que duela?** La función `experimento_desbalance` repite todo con otro factor de desbalance, y entrena el base y la cura con raíz. Prueba con **10** y con **200**. El docente corrió el barrido completo, con desbalance 1 (todas las clases con 4 500 fotos) el base sacó macro-F1 **0,871** y 91 % de camiones, con 10 sacó 0,843 y 80 %, con 100 el 0,739 y 43,5 % de hoy, y con 200, cuando camión tiene solo 22 fotos, se hundió a **0,677** y **17 %**. Lo curioso es lo que pasa con el accuracy realista, ¿sube o baja cuando el desbalance empeora?

In [ ]:
# Repite el experimento con otro factor de desbalance y compara el base con la cura de raíz
def experimento_desbalance(imb):
    # Entrenamiento y prueba realista de este desbalance
    idx_tr, n_tr = indices_entrenamiento(imb); idx_real, _ = indices_realista(imb)
    # Características del entrenamiento
    Ftr, ytr = datos_entrenamiento(idx_tr)
    # Filas de la prueba realista de este desbalance
    filas_real = torch.tensor([POS_TEST[i] for i in idx_real], device=device)
    # Encabezado
    print(f"Desbalance {imb}: {sum(n_tr)} fotos, {n_tr[0]} aviones y {n_tr[K - 1]} camiones")
    # Base y cura con raíz
    for nombre, pesos in [("base", None), ("raíz", pesos_por_clase(n_tr, 0.5))]:
        # Entrenamos
        cab, _ = entrenar_cabeza(Ftr, ytr, pesos=pesos)
        # Predecimos la prueba balanceada
        pred = puntajes_de(cab, F_TEST).argmax(1)
        # Medimos en las dos pruebas
        mb = medir(Y_TEST, pred); mr = medir(Y_TEST[filas_real], pred[filas_real])
        # Fila
        print(f"  {nombre:>5} | realista {mr['acc']:.1%} | macro-F1 {mb['f1_macro']:.3f} | recall camión {mb['recall'][K - 1]:.0%}")


# Tu predicción (escríbela antes de ejecutar):
#

# Palanca 1: prueba con 10 y con 200
experimento_desbalance(10)
experimento_desbalance(200)

**Palanca 2 · Prueba τ a mano.** Aplica el ajuste de logits sobre la prueba con τ = 0,5, 1 y 2. ¿Con cuál se cruzan los recall de avión y de camión? ¿Con cuál te pasas de la raya? En la curva de validación del docente, τ = 2 dejó a camión en 96 % y a avión en 56 %.

In [ ]:
# Tu predicción (escríbela antes de ejecutar):
#

# Palanca 2: tres valores de τ sobre el mismo modelo base
for tau in [0.5, 1.0, 2.0]:
    # Puntajes corregidos y medida (sin anotarlos en la tabla principal)
    evaluar(f"τ = {tau}", puntajes_de(cab_base, F_TEST) - tau * LOG_PRIOR)

**Palanca 3 · ¿Y si la reponderación fuera más fuerte que el inverso?** La potencia 0,5 es la raíz y la 1 es el puro. Prueba con potencia **2**, donde cada error en camión pesaría 615 y cada error en avión apenas 0,06, casi diez mil veces de diferencia (con el puro eran cien veces). El docente no corrió este caso, así que tus cifras son las primeras, y la pregunta es si más corrección siempre ayuda o si existe un punto donde la cura enferma a la cabeza.

In [ ]:
# Tu predicción (escríbela antes de ejecutar):
#

# Palanca 3: pesos con potencia 2
cab_p2, _ = entrenar_cabeza(F_TR, Y_TR, pesos=pesos_por_clase(N_TR, 2.0))
# Medida, comparable con la raíz (0,5) y el puro (1)
evaluar("potencia_2", puntajes_de(cab_p2, F_TEST))

### Pregunta escrita del laboratorio (para entregar)

Un compañero entrena un clasificador de radiografías para cinco hallazgos, uno de ellos una patología grave que aparece en el 2 % de los casos, y reporta con orgullo un **accuracy del 96 %**. En seis a ocho líneas, y **usando tus propias cifras** de las Partes 6 a 12 como ejemplo, explícale por qué ese 96 % no basta, qué dos herramientas le pedirías para saber si detecta la patología rara (y qué mirarías en cada una), y qué cura le recomendarías probar primero, diciendo qué precio va a pagar por ella.

In [ ]:
# Escribe tu respuesta aquí, como comentario:
#

---

## Cierre

Hoy pillamos al accuracy en el acto. Viste un modelo que sacó **91,0 %** en una prueba realista y apenas **74,2 %** cuando cada clase pesó lo mismo, y aprendiste a encontrar el hueco con tres herramientas, compararlo contra el modelo que siempre dice la clase más frecuente (40,4 %), mirar el **recall y la precisión de cada clase**, y leer la **matriz de confusión**, donde descubrimos que 394 de los 1 000 camiones se iban a la columna de auto. Viste que el problema no era que el cuerpo no supiera ver camiones, sino que la cabeza, ante la duda, apostaba por lo que había visto más veces. Y probaste cuatro curas, la reponderación con raíz y con inverso, el sobremuestreo y el ajuste de logits, que llevó el macro-F1 a **0,841** y el recall de camión de 43,5 % a **91 %**, siempre a costa de algo de accuracy realista.

La regla para tu proyecto es esta. Antes de entrenar, **cuenta cuántas fotos tiene cada clase**. Si la más grande tiene diez veces más que la más chica, ya estás en territorio desbalanceado. Reporta el macro-F1 y el recall por clase junto al accuracy, muestra la matriz de confusión y compara contra el modelo de la mayoría. Y elige la cura pensando en qué error cuesta más en tu caso.

**La próxima clase (Clase 12)** pasamos de preguntar "¿qué hay en esta foto?" a preguntar "¿**dónde** está cada objeto y cuántos hay?". Entramos en la **detección de objetos** con detectores tipo YOLO, cajas, IoU y umbrales de confianza. Ese umbral de confianza funciona parecido a la τ de hoy, al bajarlo encuentras más objetos y también más falsas alarmas, así que la tensión entre recall y precisión que viste hoy te va a acompañar allí también.